In [ ]:
import os
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import pickle
import psycopg2 as pyc
import itertools
import m2cgen as m2c
import shap
from sqlalchemy import create_engine
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle
from hyperopt import STATUS_OK, hp, fmin, tpe, Trials

In [ ]:
def eval_rule_hr_increase_sbp_decrease_rr_increase(params):
    # train result
    tmp = rule_hr_increase_sbp_decrease_rr_increase(params, train_label_data)

    f1_score = 0
    if tmp.shape[0] == 0:
        PPV = 0
        SE = 0
    else:
        SE = tmp[tmp['label'] == 1][['data_id', 'label_time']].drop_duplicates().shape[0]/(train_label_data[(train_label_data['label'] == 1)][['data_id', 'label_time']].drop_duplicates().shape[0])
        PPV = tmp[tmp['label'] == 1].shape[0]/(tmp.shape[0])
        TP = tmp[(tmp['label'] == 1)].shape[0]
        FP = tmp[(tmp['label'] == 0)].shape[0]
        FN = train_label_data[(train_label_data['label'] == 1)].shape[0] - TP
        print(r'train result: SE {}, PPV {}'.format(SE, PPV))
        print(r'train result: TP {}, FP {}'.format(tmp[tmp['label'] == 1].shape[0], tmp.shape[0] - tmp[tmp['label'] == 1].shape[0]))
        f1_score = (2 * TP) / (2 * TP + FP + FN)
        print(r'train result: f1_score {}'.format(f1_score))

    return tmp

In [ ]:
def train_rule_hr_increase_sbp_decrease_rr_increase(params):
    # train result
    tmp = rule_hr_increase_sbp_decrease_rr_increase(params, train_label_data)

    f1_score = 0
    PPV = 0
    SE = 0
    if tmp.shape[0] == 0:
        PPV = 0
        SE = 0
    else:
        SE = tmp[tmp['label'] == 1][['data_id', 'label_time']].drop_duplicates().shape[0]/(train_label_data[(train_label_data['label'] == 1)][['data_id', 'label_time']].drop_duplicates().shape[0])
        PPV = tmp[tmp['label'] == 1].shape[0]/(tmp.shape[0])
        TP = tmp[(tmp['label'] == 1)].shape[0]
        FP = tmp[(tmp['label'] == 0)].shape[0]
        FN = train_label_data[(train_label_data['label'] == 1)].shape[0] - TP
        f1_score = (2 * TP) / (2 * TP + FP + FN)
    
    loss = 1 - f1_score
    
    return {'loss': loss, 'params': params, 'status': STATUS_OK}

In [ ]:
# feature_data.head() = ['hr_trend_z_value',...'label','label_time']
def rule_hr_increase_sbp_decrease_rr_increase(params, feature_data):
    # hr_increase_sbp_decrease_rr_increase_rule1
    hr_ = r'(hr_trend_z_value>={} and hr_trend_time>={})'.format(params['hr_trend_z_value'], params['hr_trend_time'])
    sbp_ = r'and (sbp_trend_z_value<={} and sbp_amplitude>={} and sbp_trend_time>={})'.format(params['sbp_trend_z_value'], params['sbp_amplitude'], params['sbp_trend_time'])
    rr_ = r'and (rr_trend_z_value>={})'.format(params['rr_trend_z_value'])
    reference = r'and (hr_basic_value<th0 and sbp_basic_value>=th0)'
    end_ = r'and (sbp_trend_end_value<=th0 and rr_trend_end_value>=th0)'
    rule_id_1 = hr_ + rr_ + sbp_ + reference + end_
    # hr_increase_sbp_decrease_rr_increase_rule2
    # ...
    # rule_id_2 = hr2_ + rr2_ + sbp2_ + reference2 + end2_
    tmp = feature_data.query(rule_id_1)
    tmp['rule_hr_increase_sbp_decrease_rr_increase'] = 1

    return tmp


In [ ]:
#input the expert augemented th or default th
th1 = 0
th2 = 100
rule_params={
    'hr_trend_z_value': hp.uniform('hr_trend_z_value', th1, th2),
    'hr_trend_time': hp.uniform('hr_trend_time', th1, th2),
    'sbp_trend_z_value': hp.uniform('sbp_trend_z_value', th1, th2),
    'sbp_amplitude': hp.uniform('sbp_amplitude', th1, th2),
    'sbp_trend_time': hp.uniform('sbp_trend_time', th1, th2),
    'rr_trend_z_value': hp.uniform('rr_trend_z_value', th1, th2),
    'sbp_trend_end_value': hp.uniform('sbp_trend_end_value', th1, th2),
    'rr_trend_end_value': hp.uniform('rr_trend_end_value', th1, th2),
}
best_param = fmin(fn=train_rule_hr_increase_sbp_decrease_rr_increase, space=rule_params, algo=tpe.suggest, max_evals=200)
best_param = {x: round(y, 1) for x, y in best_params.items()}

train_hit = eval_rule_hr_increase_sbp_decrease_rr_increase(best_params)

# expert check tp,fp label with rule ,and refine the th    